# Análisis Exploratorio: Dataset Penguins (Solo Variables Numéricas)
Este notebook realiza un análisis exploratorio del dataset de pingüinos utilizando **únicamente las 4 variables numéricas**.

**Enfoque:**
- Solo variables numéricas: bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g
- Similar al análisis original de Iris
- Preparación para modelo de ML simplificado

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pandas.plotting import parallel_coordinates
from sklearn.model_selection import train_test_split

# Configuración de visualización
plt.style.use('default')
sns.set_palette('husl')
%matplotlib inline

## 1. Carga y Limpieza de Datos

In [ ]:
# Cargar el dataset
data = sns.load_dataset('penguins')
print(f"Dimensiones del dataset original: {data.shape}")

# Eliminar valores nulos
data_clean = data.dropna()
print(f"Dimensiones después de limpieza: {data_clean.shape}")
print(f"Filas eliminadas: {len(data) - len(data_clean)}")

data_clean.head(10)

In [ ]:
# Seleccionar SOLO las columnas que usaremos
# 4 variables numéricas + 1 variable objetivo (species)
columns_to_use = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'species']
data_numeric = data_clean[columns_to_use].copy()

print(f"Columnas seleccionadas: {list(data_numeric.columns)}")
print(f"\nDimensiones del dataset numérico: {data_numeric.shape}")
data_numeric.head()

## 2. Exploración Inicial

In [ ]:
# Información general
data_numeric.info()

In [ ]:
# Estadísticas descriptivas
data_numeric.describe()

In [ ]:
# Distribución de especies
print("Distribución de Especies:")
print(data_numeric['species'].value_counts())
print(f"\nTotal especies: {data_numeric['species'].nunique()}")

# Visualizar distribución
plt.figure(figsize=(8, 5))
sns.countplot(data=data_numeric, x='species', palette='Set2')
plt.title('Distribución de Especies de Pingüinos', fontsize=14, fontweight='bold')
plt.xlabel('Especie', fontsize=12)
plt.ylabel('Cantidad', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Estadísticas por Especie

In [ ]:
# Variables numéricas
numeric_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

# Estadísticas por especie
print("ESTADÍSTICAS POR ESPECIE")
print("=" * 80)
print("\nValores promedio:")
print(data_numeric.groupby('species')[numeric_cols].mean())

print("\nDesviación estándar:")
print(data_numeric.groupby('species')[numeric_cols].std())

print("\nValores mínimos:")
print(data_numeric.groupby('species')[numeric_cols].min())

print("\nValores máximos:")
print(data_numeric.groupby('species')[numeric_cols].max())

## 4. Visualizaciones - Distribuciones

In [ ]:
# Histogramas de todas las variables numéricas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    data_numeric[col].hist(bins=30, ax=axes[idx], edgecolor='black', alpha=0.7, color='skyblue')
    axes[idx].set_title(f'Distribución: {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col, fontsize=10)
    axes[idx].set_ylabel('Frecuencia', fontsize=10)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Histogramas por especie
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

species_colors = {'Adelie': '#FF6B6B', 'Chinstrap': '#4ECDC4', 'Gentoo': '#45B7D1'}

for idx, col in enumerate(numeric_cols):
    for species in data_numeric['species'].unique():
        species_data = data_numeric[data_numeric['species'] == species][col]
        axes[idx].hist(species_data, bins=20, alpha=0.6, label=species, 
                      color=species_colors[species], edgecolor='black')
    
    axes[idx].set_title(f'{col} por Especie', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(col, fontsize=10)
    axes[idx].set_ylabel('Frecuencia', fontsize=10)
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Visualizaciones - Boxplots

In [ ]:
# Boxplots por especie
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    sns.boxplot(data=data_numeric, x='species', y=col, ax=axes[idx], palette='Set2')
    axes[idx].set_title(f'{col} por Especie', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Especie', fontsize=10)
    axes[idx].set_ylabel(col, fontsize=10)
    axes[idx].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Violin plots (muestran densidad de distribución)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    sns.violinplot(data=data_numeric, x='species', y=col, ax=axes[idx], palette='muted')
    axes[idx].set_title(f'{col} por Especie (Violin Plot)', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Especie', fontsize=10)
    axes[idx].set_ylabel(col, fontsize=10)
    axes[idx].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Análisis de Correlaciones

In [ ]:
# Matriz de correlación
correlation_matrix = data_numeric[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=2, cbar_kws={"shrink": 0.8},
            fmt='.3f', annot_kws={'size': 11})
plt.title('Matriz de Correlación - Variables Numéricas', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("Matriz de Correlación:")
print(correlation_matrix)

In [ ]:
# Identificar correlaciones más fuertes
print("CORRELACIONES MÁS FUERTES:")
print("=" * 60)

# Crear lista de correlaciones (excluyendo diagonal)
correlations = []
for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        correlations.append({
            'Variable 1': numeric_cols[i],
            'Variable 2': numeric_cols[j],
            'Correlación': correlation_matrix.iloc[i, j]
        })

corr_df = pd.DataFrame(correlations).sort_values('Correlación', ascending=False, key=abs)
print(corr_df.to_string(index=False))

## 7. Análisis Multivariado - Pairplot

In [ ]:
# Pairplot: Similar al análisis de Iris
sns.pairplot(data_numeric, hue='species', diag_kind='kde', height=2.5, 
             palette='Set1', plot_kws={'alpha': 0.6})
plt.suptitle('Pairplot: Relaciones entre Variables Numéricas', y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Pairplot alternativo con histogramas en diagonal
sns.pairplot(data_numeric, hue='species', diag_kind='hist', height=2.5, 
             palette='deep', plot_kws={'alpha': 0.7, 's': 50, 'edgecolor': 'black'})
plt.suptitle('Pairplot con Histogramas', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## 8. Gráficos de Dispersión Clave

In [ ]:
# Gráficos de dispersión más relevantes
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

# Pares de variables a comparar
pairs = [
    ('bill_length_mm', 'bill_depth_mm'),
    ('bill_length_mm', 'flipper_length_mm'),
    ('bill_length_mm', 'body_mass_g'),
    ('bill_depth_mm', 'flipper_length_mm'),
    ('bill_depth_mm', 'body_mass_g'),
    ('flipper_length_mm', 'body_mass_g')
]

for idx, (var1, var2) in enumerate(pairs):
    sns.scatterplot(data=data_numeric, x=var1, y=var2, hue='species', 
                    s=80, ax=axes[idx], palette='bright', alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'{var1} vs {var2}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel(var1.replace('_', ' ').title(), fontsize=9)
    axes[idx].set_ylabel(var2.replace('_', ' ').title(), fontsize=9)
    axes[idx].legend(title='Especie', fontsize=8, title_fontsize=9)
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Análisis de Separabilidad de Especies

In [ ]:
# Visualización enfocada: ¿Qué variable separa mejor las especies?
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    # Swarm plot muestra todos los puntos individuales
    sns.swarmplot(data=data_numeric, x='species', y=col, ax=axes[idx], 
                  palette='Set2', size=4, alpha=0.7)
    axes[idx].set_title(f'Separabilidad por {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Especie', fontsize=10)
    axes[idx].set_ylabel(col, fontsize=10)
    axes[idx].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Coordenadas Paralelas

In [ ]:
# Coordenadas paralelas (como en Iris)
plt.figure(figsize=(14, 7))
parallel_coordinates(data_numeric, 'species', color=['#FF6B6B', '#4ECDC4', '#45B7D1'], linewidth=1.5)
plt.title('Coordenadas Paralelas - Todas las Variables Numéricas', fontsize=14, fontweight='bold')
plt.xlabel('Variables', fontsize=12)
plt.ylabel('Valores Normalizados', fontsize=12)
plt.legend(title='Especie', loc='best', fontsize=10)
plt.xticks(rotation=20)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Preparación para Machine Learning

In [ ]:
# División de datos: 60% entrenamiento, 40% prueba
# Usar stratify para mantener proporciones de especies
train, test = train_test_split(data_numeric, test_size=0.4, 
                                stratify=data_numeric['species'], 
                                random_state=42)

print(f"Total de datos: {len(data_numeric)}")
print(f"Datos de entrenamiento: {len(train)} ({len(train)/len(data_numeric)*100:.1f}%)")
print(f"Datos de prueba: {len(test)} ({len(test)/len(data_numeric)*100:.1f}%)")

print("\n" + "=" * 60)
print("Distribución de especies en ENTRENAMIENTO:")
print(train['species'].value_counts())
print(f"\nProporciones: {train['species'].value_counts(normalize=True)}")

print("\n" + "=" * 60)
print("Distribución de especies en PRUEBA:")
print(test['species'].value_counts())
print(f"\nProporciones: {test['species'].value_counts(normalize=True)}")

In [ ]:
# Visualizar la división
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conjunto de entrenamiento
train['species'].value_counts().plot(kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4', '#45B7D1'], alpha=0.8)
axes[0].set_title('Distribución - Conjunto de Entrenamiento', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Especie', fontsize=10)
axes[0].set_ylabel('Cantidad', fontsize=10)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(alpha=0.3, axis='y')

# Conjunto de prueba
test['species'].value_counts().plot(kind='bar', ax=axes[1], color=['#FF6B6B', '#4ECDC4', '#45B7D1'], alpha=0.8)
axes[1].set_title('Distribución - Conjunto de Prueba', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Especie', fontsize=10)
axes[1].set_ylabel('Cantidad', fontsize=10)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Visualización de coordenadas paralelas solo para entrenamiento
plt.figure(figsize=(14, 7))
parallel_coordinates(train, 'species', color=['#FF6B6B', '#4ECDC4', '#45B7D1'], linewidth=1.5, alpha=0.7)
plt.title('Coordenadas Paralelas - Conjunto de Entrenamiento', fontsize=14, fontweight='bold')
plt.xlabel('Variables', fontsize=12)
plt.ylabel('Valores Normalizados', fontsize=12)
plt.legend(title='Especie', loc='best', fontsize=10)
plt.xticks(rotation=20)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Resumen del Análisis Exploratorio

In [ ]:
print("=" * 80)
print("RESUMEN DEL ANÁLISIS EXPLORATORIO")
print("=" * 80)

print(f"\n1. DATOS:")
print(f"   - Total de registros (sin nulos): {len(data_numeric)}")
print(f"   - Variables numéricas: {len(numeric_cols)}")
print(f"   - Especies: {data_numeric['species'].nunique()}")

print(f"\n2. VARIABLES UTILIZADAS:")
for i, col in enumerate(numeric_cols, 1):
    print(f"   {i}. {col}")

print(f"\n3. DISTRIBUCIÓN DE ESPECIES:")
for species, count in data_numeric['species'].value_counts().items():
    percentage = (count / len(data_numeric)) * 100
    print(f"   - {species}: {count} ({percentage:.1f}%)")

print(f"\n4. CORRELACIONES MÁS FUERTES:")
top_3_corr = corr_df.head(3)
for _, row in top_3_corr.iterrows():
    print(f"   - {row['Variable 1']} vs {row['Variable 2']}: {row['Correlación']:.3f}")

print(f"\n5. OBSERVACIONES CLAVE:")
print(f"   - Flipper length y body mass están altamente correlacionados")
print(f"   - Gentoo es claramente la especie más grande")
print(f"   - Adelie tiene el bill más profundo pero más corto")
print(f"   - Las especies son separables usando solo variables numéricas")

print(f"\n6. CONJUNTOS DE DATOS PARA ML:")
print(f"   - Entrenamiento: {len(train)} muestras ({len(train)/len(data_numeric)*100:.1f}%)")
print(f"   - Prueba: {len(test)} muestras ({len(test)/len(data_numeric)*100:.1f}%)")

print("\n" + "=" * 80)
print("LISTO PARA ENTRENAR MODELO DE MACHINE LEARNING")
print("=" * 80)

In [ ]:
# Guardar los conjuntos de datos para el siguiente notebook
train.to_csv('penguins_train_numeric.csv', index=False)
test.to_csv('penguins_test_numeric.csv', index=False)

print("Datos guardados:")
print("  - penguins_train_numeric.csv")
print("  - penguins_test_numeric.csv")
print(f"\nEstos archivos contienen solo las {len(numeric_cols)} variables numéricas + species")